In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kholoudabdelmohsen/cleaned-data/train(1).csv


**CSV DATA**

In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv("/kaggle/input/datasets/kholoudabdelmohsen/cleaned-data/train(1).csv")

# Drop 2nd, 4th, and 5th columns
df = df.drop(df.columns[[1, 3, 4]], axis=1)

# Show the first rows
df.head()

,label,cleaned_text
0,Fungal infection,ive been quite itchy recently and i have rashy...
1,Pneumonia,i have a really high fever and im having troub...
2,Varicose Veins,standing or walking for long periods of time h...
3,Malaria,ive been experiencing severe itching chills vo...
4,Typhoid,most of the time i feel fatigued i dont want t...


In [3]:
text_data = []

for _, row in df.iterrows():
    text = " | ".join([f"{col}: {row[col]}" for col in df.columns])
    text_data.append(text)

print(text_data[0])

label: Fungal infection | cleaned_text: ive been quite itchy recently and i have rashy patches all over my skin there are also certain regions that are darker in colour than the rest of my skin and ive got some firm lumps


In [6]:
from langchain_core.documents import Document

csv_docs = [
    Document(page_content=text, metadata={"source": "csv"})
    for text in text_data
]

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [27]:
csv_chunks = text_split(csv_docs)
print(len(csv_chunks))

960


In [15]:
csv_chunks[1]

Document(metadata={'source': 'csv'}, page_content='label: Pneumonia | cleaned_text: i have a really high fever and im having trouble breathing im sweating a lot and feeling really cold and tired my heart is beating really fast and i have some brownish sputum')

In [7]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>

BOOK DATA****

In [8]:
!pip install pypdf

In [9]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
import os
import gdown
import sqlite3


In [11]:
!gdown --folder https://drive.google.com/drive/folders/1JzWXDm2hGMaVQJxwt7L4RfPL2ukFVNsI  #download the data which is the book

Retrieving folder contents
Processing file 1w9axjggiUfU5fpRaa5AZH3dZb5f7M9sp Medical_book.pdf
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1w9axjggiUfU5fpRaa5AZH3dZb5f7M9sp
To: /kaggle/working/depi data/Medical_book.pdf
100%|███████████████████████████████████████| 16.1M/16.1M [00:00<00:00, 146MB/s]
Download completed


In [13]:
def load_pdf_files(file_path):
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    return documents
    documents = loader.load()
    return documents

In [14]:
extracted_data = load_pdf_files(r"/kaggle/working/depi data/Medical_book.pdf") #retrieve a list at which each element in the list is an lang chain document

In [15]:
extracted_data[200]

Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': '/kaggle/working/depi data/Medical_book.pdf', 'total_pages': 637, 'page': 200, 'page_label': '201'}, page_content='• Nitrous oxide (laughing gas) is a weak anesthetic and is\nused with other agents, such as thiopental, to produce\nsurgical anesthesia. It has the fastest induction and\nrecovery and is the safest because it does not slow\nbreathing or blood flow to the brain. However, it diffus-\nes rapidly into air-containing cavities and can result in a\ncollapsed lung ( pneumothorax ) or lower the oxygen\ncontents of tissues (hypoxia).\nCommonly administered intravenous anesthetic\nagents include ketamine, thiopental, opioids, and propofol.\n• Ketamine (Ketalar) affects the senses, and produces a\ndissociative anesthesia (catatonia, amnesia, analgesia)\nin which the patient may appear awake and reactive, but\ncann

In [16]:
len(extracted_data)   #we have retrieved each page separtly and saved it in a list

637

In [17]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Keep only:
    - page_content
    - source
    - page number
    """
    minimal_docs: List[Document] = []

    for doc in docs:
        src = doc.metadata.get("source")
        page = doc.metadata.get("page")

        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={
                    "source": src,
                    "page": page
                }
            )
        )

    return minimal_docs

In [19]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [21]:
minimal_docs[200]

Document(metadata={'source': '/kaggle/working/depi data/Medical_book.pdf', 'page': 200}, page_content='• Nitrous oxide (laughing gas) is a weak anesthetic and is\nused with other agents, such as thiopental, to produce\nsurgical anesthesia. It has the fastest induction and\nrecovery and is the safest because it does not slow\nbreathing or blood flow to the brain. However, it diffus-\nes rapidly into air-containing cavities and can result in a\ncollapsed lung ( pneumothorax ) or lower the oxygen\ncontents of tissues (hypoxia).\nCommonly administered intravenous anesthetic\nagents include ketamine, thiopental, opioids, and propofol.\n• Ketamine (Ketalar) affects the senses, and produces a\ndissociative anesthesia (catatonia, amnesia, analgesia)\nin which the patient may appear awake and reactive, but\ncannot respond to sensory stimuli. These properties\nmake it especially useful for use in developing coun-\ntries and during warfare medical treatment. Ketamine is\nfrequently used in pediat

In [22]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,                                              #craetion of chunks each chunck has 500 character and overlap with 20 characters so we donot lose the contextual meaning
        chunk_overlap=50,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [23]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 5961


In [24]:
texts_chunk[500]

Document(metadata={'source': '/kaggle/working/depi data/Medical_book.pdf', 'page': 60}, page_content='an abscess may be necessary.\nPrognosis\nComplete recovery is expected if antibiotic treatment\nis begun at an early stage of the infection. However, if\nuntreated, acute lymphangitis can be a very serious and\neven deadly disease. Acute lymphangitis that goes untreat-\ned can spread, causing tissue damage. Extensive tissue\ndamage would need to be repaired by plastic surgery.\nSpread of the infection into the bloodstream could be fatal.\nPrevention\nAlthough acute lymphangitis can occur in anyone, good')

In [28]:
final_chunks = texts_chunk + csv_chunks

In [29]:
len(final_chunks)

6921

In [30]:
final_chunks[5000]

Document(metadata={'source': '/kaggle/working/depi data/Medical_book.pdf', 'page': 534}, page_content='blood cells). Technologists then examine a stained blood\nsmear under the microscope to identify any abnormali-\nties in the appearance of the red blood cells and to report\nthe types and percentages of white blood cells observed.\nThe red blood cell (RBC) count determines the total\nnumber of red cells (erythrocytes) in a sample of blood.\nThe red cells, the most numerous of the cellular ele-\nments, carry oxygen from the lungs to the body’s tissues.')

In [31]:
final_chunks[6000]

Document(metadata={'source': 'csv'}, page_content='label: Typhoid | cleaned_text: i am having a terrible pain in my abdominal part and ive been feeling really nauseated im also experiencing a mild fever i am really worried')

In [32]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    
    embeddings = HuggingFaceEmbeddings(                                #converting data into numerical vector chosed this one as it is of low storage , and semantic understanding
        model_name=model_name
    )
    
    return embeddings

embedding = download_embeddings()

/tmp/ipykernel_57/831715917.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(                                #converting data into numerical vector chosed this one as it is of low storage , and semantic understanding


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [33]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [34]:
import os
import json

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

os.environ['hahakh'] = secrets.get_secret("hahakh")
os.environ['yarab'] = secrets.get_secret("yarab")
# Change below
meta = dict(
    id="ks/RAGdata",
    title="medical_data",
    isPrivate=False,
    licenses=[dict(name="other")]
)

with open('/kaggle/working/dataset-metadata.json', 'w') as f:
    json.dump(meta, f)

In [35]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.8 MB/s eta 0:00:00:00:0100:01


In [36]:
import sqlite3
import uuid
from langchain_community.embeddings import HuggingFaceEmbeddings
import uuid
from langchain_community.vectorstores import FAISS

In [37]:
def create_vectorstore(chunks, embedding_function):
    unique_chunks = []
    unique_ids = set()

    for doc in chunks:
        # Include metadata if available
        source = doc.metadata.get("source", "")
        row_id = doc.metadata.get("row", "")
        page = doc.metadata.get("page", "")   # <-- page number added
        
        # Create a more robust unique string
        unique_string = f"{source}_{page}_{row_id}_{doc.page_content}"

        # Generate UUID
        doc_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, unique_string))

        # Deduplicate
        if doc_id not in unique_ids:
            unique_ids.add(doc_id)
            unique_chunks.append(doc)

    # Create FAISS vectorstore
    vectorstore = FAISS.from_documents(
        documents=unique_chunks,
        embedding=embedding_function
    )

    return vectorstore

In [38]:
# Create and persist the vectorstore
vectorstore = create_vectorstore(chunks=final_chunks, embedding_function=embedding)

In [39]:
print(vectorstore.index.ntotal)

6890


In [40]:
first_vector = vectorstore.index.reconstruct(6000)
print(first_vector)

[ 8.64011422e-02 -1.14000915e-03  5.54722399e-02 -2.88781859e-02
  1.66889355e-02 -8.19098055e-02  9.51168314e-02  1.13564871e-01
  1.16616143e-02 -1.45481713e-02 -1.59072056e-02 -5.90829225e-03
  8.15329626e-02  5.40242307e-02 -3.08439452e-02 -4.96469475e-02
  7.69857988e-02 -6.60333261e-02  8.47487431e-03  1.83609240e-02
 -1.02676190e-01 -1.06429579e-02  6.27854886e-03 -4.42336723e-02
  4.07394394e-02  5.31992130e-02 -6.78654900e-03  3.70915160e-02
 -4.63641137e-02 -2.54511070e-02 -5.52020334e-02  4.69029285e-02
  1.74082164e-02 -7.44711561e-03  4.83255535e-02  1.67945847e-02
 -3.19751315e-02 -1.81199834e-02  4.83088791e-02 -1.11507615e-02
  2.61479858e-02 -2.83542555e-03  3.55796912e-03 -1.30296936e-02
  2.11392231e-02  4.25366871e-02 -5.51567003e-02  4.57027256e-02
  8.00596327e-02  9.41567943e-02 -2.73933001e-02 -6.95311576e-02
  9.11194831e-02  5.85001782e-02 -2.90911365e-02  1.28160696e-02
 -2.41075270e-02 -1.01582088e-01 -8.77399370e-02 -5.75276054e-02
 -8.88466761e-02 -2.15384

In [41]:
vectorstore.save_local("/kaggle/working/faiss_index")

In [42]:
!zip -r faiss_index.zip /kaggle/working/faiss_index

  adding: kaggle/working/faiss_index/ (stored 0%)
  adding: kaggle/working/faiss_index/index.faiss (deflated 7%)
  adding: kaggle/working/faiss_index/index.pkl (deflated 64%)
